# P3.5 DeDo — Down + DSL-x6, optionally FPN.A-oA

One notebook, select the experiment in the first config cell. `ex0` is the existing trained DeDo-alx6.3Aall; five additional modes combine LIP-Early/DPP-All/AAA-All with DSL-x6 and optionally A-oA. A-oA retains the FPN campaign hidden widths. The auxiliary head is training-only; downsampling and feature fusion remain at inference.

Flow retained: clone f/deepdown → attach PhenoBench → GPU T4×2 → Save & Run All → one ZIP. P2 recipe: 100 epochs / validation every 5, same seed, FP32, WDice, optimizer, batch and augmentations. D01/D02/D03 guards retained. Publish the prepared f/deepdown source before running on Kaggle.


In [ ]:
# FIRST CONFIG CELL
REPO_URL = 'https://github.com/FUAVX/RPD_Reproduce.git'
GIT_REF = 'f/deepdown'
SOURCE_COMMIT = ''  # optional commit reachable from this branch
# Select a key from EXPERIMENTS below; ex0 preserves trained DeDo-alx6.3Aall.
EXPERIMENT_MODE = 'ex0'
RUN_TAG = 'run1'
DATASET_DIR = ''  # empty: detect attached PhenoBench
NUM_WORKERS = 2
DEVICES = 2
WORKING_ROOT = '/kaggle/working'
RUN_BENCHMARKS = True  # disposable control/selected mode, established project protocol
EXPORT_MAIN = True  # main-only export retains downsampling/fusion and strips aux head
EXPERIMENTS = {'control': ('DeepDown-Control', 'DeepDown_Control', None),
 'ex0': ('DeepDown-Ex0', 'DeepDown_Ex0', 'x6'),
 'alx6.LipEar': ('DeDo-alx6.LipEar', 'DeDo_alx6_LipEar', 'x6'),
 'alx6.DppAll': ('DeDo-alx6.DppAll', 'DeDo_alx6_DppAll', 'x6'),
 'alx6.3Aall.elwAdd': ('DeDo-alx6.3Aall.elwAdd', 'DeDo_alx6_3Aall_elwAdd', 'x6'),
 'alx6.LipEar.elwAdd': ('DeDo-alx6.LipEar.elwAdd', 'DeDo_alx6_LipEar_elwAdd', 'x6'),
 'alx6.DppAll.elwAdd': ('DeDo-alx6.DppAll.elwAdd', 'DeDo_alx6_DppAll_elwAdd', 'x6')}
if EXPERIMENT_MODE not in EXPERIMENTS:
    raise ValueError(f'Select one of {list(EXPERIMENTS)}')
if DEVICES != 2:
    raise ValueError('P2 runtime protocol requires T4 x2')
SCENARIO, CONFIG_STEM, EXPECTED_TAP = EXPERIMENTS[EXPERIMENT_MODE]


In [ ]:
# Kaggle supplies torch/CUDA; retain that installation.
!nvidia-smi
!pip install -q 'numpy<2' 'pytorch-lightning==1.9.5' 'torchmetrics==0.11.4' 'setuptools<70' tensorboard matplotlib pandas tabulate timm scikit-image tifffile albumentations pyyaml rich


In [ ]:
import json
import pathlib
import shutil
import subprocess
import yaml
WORKING_DIR = pathlib.Path(WORKING_ROOT)
REPO_DIR = WORKING_DIR / 'RPD_P2'
RUN_DIR = WORKING_DIR / f'{SCENARIO}_{RUN_TAG}'
if RUN_DIR.exists():
    raise FileExistsError(f'Output already exists: {RUN_DIR}; choose a new RUN_TAG')
def run(command, cwd=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run(command, cwd=cwd, check=True)
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
run(['git', 'clone', '--branch', GIT_REF, '--single-branch', REPO_URL, str(REPO_DIR)])
if SOURCE_COMMIT:
    run(['git', 'checkout', '--detach', SOURCE_COMMIT], cwd=REPO_DIR)
BASE_COMMIT = 'd77d518b8f650443216070fa76742db76e41b8ac'
run(['git', 'merge-base', '--is-ancestor', BASE_COMMIT, 'HEAD'], cwd=REPO_DIR)
PARENT_COMMITS = {'DeepSup': '3100c6fd1f2bb9fe3887f2b15a0a306054e21c38',
                  'AAA': '0d2f5c55299a297feae3aecdbe9d2a8ae276e93d'}
for parent_commit in PARENT_COMMITS.values():
    run(['git', 'merge-base', '--is-ancestor', parent_commit, 'HEAD'], cwd=REPO_DIR)
scenario_path = REPO_DIR / 'configs/p35_deepdown' / f'{CONFIG_STEM}.yaml'
required_paths = [scenario_path] + [REPO_DIR / name for name in (
    'test_dedo.py', 'utils/deepdown_campaign.py', 'models/rpdnet/fuse_fpn.py', 'test_wdice.py', 'test_aaa.py', 'test_deepdown.py', 'test_deepsup.py', 'test_validation_infrastructure.py',
    'models/rpdnet/aaa.py', 'models/rpdnet/downsampling.py', 'tests/reference/aaa/blurpool.py',
    'utils/validation_artifacts.py', 'scripts/check_deepsup_run.py',
    'models/rpdnet/deep_supervision.py',
    'scripts/benchmark_deepsup.py', 'scripts/export_deepsup.py', 'scripts/check_deepsup_data.py',
    'callbacks/deepsup_cost.py', 'utils/tensorboard_compat.py', 'tests/reference/deepsup/pspnet.py')]
missing = [str(path) for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError('DeepDown source files are missing: ' + ', '.join(missing))
def is_phenobench_root(path):
    return (
        (path / 'train' / 'images').is_dir()
        and (path / 'train' / 'semantics').is_dir()
        and (path / 'val' / 'images').is_dir()
        and (path / 'val' / 'semantics').is_dir()
    )

if DATASET_DIR:
    dataset_dir = pathlib.Path(DATASET_DIR)
else:
    kaggle_input = pathlib.Path('/kaggle/input')
    candidates = [
        kaggle_input / 'datasets' / 'ndhoang2310' / 'phenobench-dataset' / 'PhenoBench',
        kaggle_input / 'phenobench-dataset' / 'PhenoBench',
        kaggle_input / 'datasets' / 'ndhoang2310' / 'phenobench-dataset',
        kaggle_input / 'phenobench-dataset',
    ]
    if kaggle_input.is_dir():
        attached_inputs = [p for p in kaggle_input.iterdir() if p.is_dir()]
        candidates.extend(p / 'PhenoBench' for p in attached_inputs)
        candidates.extend(attached_inputs)
    dataset_dir = next((p for p in candidates if is_phenobench_root(p)), None)
if dataset_dir is None or not is_phenobench_root(dataset_dir):
    raise FileNotFoundError('Cannot find a complete PhenoBench root in /kaggle/input. Attach the PhenoBench dataset, or set DATASET_DIR to its root containing train/images, train/semantics, val/images, and val/semantics.')

print(f'[OK] source ref: {subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()}')
print(f'[OK] dataset: {dataset_dir}')
print(f'[OK] train images: {len(list((dataset_dir / "train" / "images").glob("*")))}')


In [ ]:
# Runtime overrides follow the successful WDice notebook.
import copy
import hashlib
RUN_DIR.mkdir(parents=True, exist_ok=True)
cfg = yaml.safe_load(scenario_path.read_text())
base_path = REPO_DIR / 'configs/p2_scenarios/P2_Ex7_RepDWNet_WDice-Loss.yaml'
base_cfg = yaml.safe_load(base_path.read_text())
inherited = copy.deepcopy(cfg)
settings = inherited['backbone'].pop('deep_supervision', None)
downs = inherited['backbone'].pop('downsampling', None)
fusion = inherited['backbone'].pop('fusion', None)
inherited['experiment'] = base_cfg['experiment']
assert inherited == base_cfg, 'Recipe drift beyond metadata/downsampling/DSL-x6/A-oA'
import sys
sys.path.insert(0, str(REPO_DIR))
from utils.deepdown_campaign import EXPERIMENTS as SOURCE_EXPERIMENTS
assert EXPERIMENTS == SOURCE_EXPERIMENTS, 'Notebook/source catalog mismatch'
assert settings is None if EXPECTED_TAP is None else settings == {'tap': 'x6', 'weight': 0.4, 'hidden_ratio': 0.5, 'dropout': 0.1, 'align_corners': True}
assert (fusion == {'variant': 'add'}) == EXPERIMENT_MODE.endswith('.elwAdd')
assert cfg['experiment']['id'] == SCENARIO
if EXPECTED_TAP:
    down_name = '3Aall' if EXPERIMENT_MODE == 'ex0' else EXPERIMENT_MODE.split('.')[1]
    parent_stem = {'3Aall': 'P3_AAA_Ex3_All', 'LipEar': 'P3_LIP_Ex1_Early',
                   'DppAll': 'P3_DPP_Ex3_All'}[down_name]
    down_parent = yaml.safe_load((REPO_DIR / 'configs/p3_scenarios' / f'{parent_stem}.yaml').read_text())
    assert downs == down_parent['backbone']['downsampling'], 'Downsampling recipe drift'
else:
    assert downs is None
assert cfg['train']['max_epoch'] == 100
assert cfg['val']['check_val_every_n_epoch'] == 5
assert cfg['train']['precision'] == 32 and cfg['train']['batch_size'] == 4
assert cfg['train']['loss'] == 'weighted_dice'
cfg['data']['path_to_dataset'] = str(dataset_dir)
cfg['data']['num_workers'] = NUM_WORKERS
active_cfg_path = RUN_DIR / f'active_config_{SCENARIO}.yaml'
active_cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
runtime_cfg = copy.deepcopy(cfg)
runtime_cfg['train']['devices'] = DEVICES
runtime_cfg['val']['devices'] = 1  # exact legacy --devices override
runtime_cfg_path = RUN_DIR / 'runtime_config.yaml'
runtime_cfg_path.write_text(yaml.safe_dump(runtime_cfg, sort_keys=False))
print(yaml.safe_dump(runtime_cfg, sort_keys=False))


In [ ]:
# GPU and data guards run before any campaign training.
import torch
if not torch.cuda.is_available() or torch.cuda.device_count() != DEVICES:
    raise RuntimeError('Choose Kaggle GPU T4 x2')
gpu_names = [torch.cuda.get_device_name(i) for i in range(DEVICES)]
if any('T4' not in name for name in gpu_names):
    raise RuntimeError(f'Expected T4 x2, got {gpu_names}')
run(['python', 'scripts/check_deepsup_data.py', '--dataset-dir', str(dataset_dir),
     '--output', str(RUN_DIR / 'dataset_audit.json')], cwd=REPO_DIR)
for script in ['test_wdice.py', 'test_validation_infrastructure.py', 'test_aaa.py', 'test_deepdown.py', 'test_dedo.py']:
    run(['python', script], cwd=REPO_DIR)
infrastructure_smoke = ['python', 'test_validation_infrastructure.py', '--ddp',
    '--output-dir', str(RUN_DIR / 'smoke/validation_infrastructure'),
    '--accelerator', 'gpu', '--devices', str(DEVICES)]
run(infrastructure_smoke, cwd=REPO_DIR)
run(infrastructure_smoke + ['--resume'], cwd=REPO_DIR)
for mode in dict.fromkeys(['control', EXPERIMENT_MODE]):
    scenario, config_stem, tap = EXPERIMENTS[mode]
    smoke = ['python', 'test_deepdown.py', '--smoke', '--config',
             str(REPO_DIR / 'configs/p35_deepdown' / f'{config_stem}.yaml'),
             '--output-dir', str(RUN_DIR / 'smoke' / mode), '--accelerator', 'gpu',
             '--devices', str(DEVICES)]
    run(smoke, cwd=REPO_DIR)
    run(smoke + ['--resume'], cwd=REPO_DIR)
run(['python', 'test_deepdown.py', '--pipeline-smoke', '--config', str(scenario_path),
     '--output-dir', str(RUN_DIR / 'smoke/pipeline'), '--accelerator', 'gpu',
     '--devices', str(DEVICES)], cwd=REPO_DIR)
# Keep full CLI smoke evidence nested so generic artifact readers see only
# the real campaign training_output/val_history.csv and selected checkpoints.
shutil.make_archive(str(RUN_DIR / 'smoke/pipeline_evidence'), 'zip', root_dir=RUN_DIR / 'smoke/pipeline')
shutil.rmtree(RUN_DIR / 'smoke/pipeline')
run(['python', 'test_deepdown.py', '--native-smoke', '--config', str(scenario_path), '--accelerator', 'gpu',
     '--output-dir', str(RUN_DIR / 'smoke/native')], cwd=REPO_DIR)
if RUN_BENCHMARKS:
    # Disposable processes: no weights/BN/optimizer/RNG carry into the campaign.
    run(['python', '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2',
         'scripts/benchmark_deepsup.py', '--campaign', 'deepdown', '--experiments', 'control', EXPERIMENT_MODE, '--mode', 'training', '--device', 'cuda',
         '--output-dir', str(RUN_DIR / 'benchmark')], cwd=REPO_DIR)
    run(['python', 'scripts/benchmark_deepsup.py', '--campaign', 'deepdown', '--experiments', 'control', EXPERIMENT_MODE, '--mode', 'inference', '--device', 'cuda',
         '--output-dir', str(RUN_DIR / 'benchmark')], cwd=REPO_DIR)
TRAIN_LOG_DIR = RUN_DIR / 'training_output'
train_command = ['python', 'multi_metric_train.py', '--config', str(active_cfg_path),
                 '--export_dir', str(TRAIN_LOG_DIR), '--devices', str(DEVICES)]
run(train_command, cwd=REPO_DIR)


In [ ]:
# Require checkpoint/history/monitor parity before exports or completed manifest.
run(['python', 'scripts/check_deepsup_run.py', '--output-dir', str(TRAIN_LOG_DIR),
     '--report', str(RUN_DIR / 'campaign_audit.json')], cwd=REPO_DIR)
# Export main-only checkpoints separately. Fusion is an explicit later audit.
if EXPORT_MAIN:
    for policy in ['best_mIoU', 'best_weedIoU']:
        matches = sorted((TRAIN_LOG_DIR / 'checkpoints').glob(f'*{policy}*.ckpt'))
        if len(matches) != 1:
            raise RuntimeError(f'Expected exactly one {policy} checkpoint: {matches}')
        run(['python', 'scripts/export_deepsup.py', '--config', str(runtime_cfg_path),
             '--checkpoint', str(matches[0]), '--output-dir', str(RUN_DIR / 'main_export' / policy),
             '--device', 'cuda'], cwd=REPO_DIR)
import importlib.metadata
source_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
manifest = {
    'experiment': SCENARIO, 'mode': EXPERIMENT_MODE, 'tap': EXPECTED_TAP, 'run_tag': RUN_TAG,
    'source_repository': REPO_URL, 'requested_ref': GIT_REF, 'requested_commit': SOURCE_COMMIT,
    'source_commit': source_commit, 'base_commit': BASE_COMMIT, 'parent_commits': PARENT_COMMITS,
    'downsampling': downs, 'feature_fusion': fusion,
    'FPN_A_oA_source': {'branch': 'f/fuse-fpn', 'commit': 'aa17f134d7bd26d88c0d4efce948f91425ac4932',
        'mid_channels': [128, 64, 32, 16]} if fusion else None, 'deep_reference_mode': 'DeepSup-Ex4-x6' if EXPECTED_TAP else None,
    'base_config': str(base_path.relative_to(REPO_DIR)),
    'base_config_sha256': hashlib.sha256(base_path.read_bytes()).hexdigest(),
    'resolved_config': runtime_cfg, 'train_command': train_command,
    'batch_size_per_GPU': cfg['train']['batch_size'], 'global_batch_size': cfg['train']['batch_size'] * DEVICES,
    'gpu_names': gpu_names, 'bn': 'standard BatchNorm2d, sync_batchnorm=False',
    'seed': cfg['seed'], 'same_production_criterion': True, 'main_only_validation': True,
    'auxiliary': settings, 'inference_downsampling_retained': bool(downs), 'inference_feature_fusion_retained': bool(fusion), 'inference_AAA_retained': (downs or {}).get('variant') == 'aaa_tri3', 'head_initialization': 'PyTorch layer defaults, after host in CPU RNG fork/restore; no new seed',
    'AAA_reference': {'repo': 'https://github.com/adobe/antialiased-cnns',
        'commit': 'b27a34a26f3ab039113d44d83c54d0428598ac9c',
        'implementation': 'models/rpdnet/aaa.py; fixed Tri-3 MaxBlurPool'} if (downs or {}).get('variant') == 'aaa_tri3' else None,
    'DPP_reference': {'repo': 'https://github.com/visinf/dpp',
        'commit': 'e810bd63ed61bd7710559467bcd9a35ab9a1b028',
        'implementation': 'models/rpdnet/dpp.py; existing Lite-DPPsym'} if (downs or {}).get('variant') == 'lite_dpp_sym' else None,
    'LIP_reference': {'repo': 'https://github.com/sebgao/LIP',
        'commit': '6724ec63c1f1ebefa25d7269e29d17c4411a7941',
        'implementation': 'models/rpdnet/lip.py; existing C/4 bottleneck'} if (downs or {}).get('variant') == 'lip_bottleneck' else None,
    'official_reference': {'repo': 'https://github.com/hszhao/semseg',
        'commit': '4f274c3f276778228bc14a4565822d46359f0cc8', 'file': 'model/pspnet.py'},
    'benchmarks_requested': RUN_BENCHMARKS, 'main_export_requested': EXPORT_MAIN,
    'versions': {name: importlib.metadata.version(name) for name in
                 ('torch', 'torchvision', 'pytorch-lightning', 'torchmetrics', 'numpy')},
    'validation_infrastructure': {'D01': 'preflight stops on 255; metric semantics unchanged',
        'D02': 'global synchronized metrics; rank-zero persistent writer',
        'D03': 'checkpoint-owned validation history; no sanity rows; epoch deduplication'},
    'validation_sampling': {'drop_last': True, 'batch_size_per_rank': cfg['val']['batch_size'],
        'world_size': DEVICES, 'note': 'P2 sampling retained; independent full validation evaluation is separate'},
    'dataset_audit': json.loads((RUN_DIR / 'dataset_audit.json').read_text()),
    'campaign_audit': json.loads((RUN_DIR / 'campaign_audit.json').read_text()),
    'status': 'completed',
}
required_outputs = [TRAIN_LOG_DIR / 'checkpoints/last.ckpt', TRAIN_LOG_DIR / 'val_history.csv',
                    TRAIN_LOG_DIR / 'training_cost.json', runtime_cfg_path, RUN_DIR / 'dataset_audit.json',
                    RUN_DIR / 'campaign_audit.json']
if (downs or {}).get('variant') == 'lite_dpp_sym':
    required_outputs.extend(TRAIN_LOG_DIR / 'dpp_parameters' / name
                            for name in ['initial.json', 'final.json', 'history.jsonl'])
if EXPECTED_TAP:
    required_outputs.append(TRAIN_LOG_DIR / 'deep_supervision_history.csv')
if EXPORT_MAIN:
    required_outputs.extend(RUN_DIR / 'main_export' / policy / 'main_state_dict.pt'
                            for policy in ['best_mIoU', 'best_weedIoU'])
missing = [str(path) for path in required_outputs if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing completed artifacts: ' + ', '.join(missing))
(RUN_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2))
archive = shutil.make_archive(str(WORKING_DIR / f'{SCENARIO}_{RUN_TAG}_artifacts'), 'zip', root_dir=RUN_DIR)
print(f'[DONE] Download: {archive}')
print(f'[DONE] Source: {source_commit}')
